# Lists, Arrays, and a First Spectrum
### 1.4 — Foundations · Beginner · 12–14 min

A spectrum or a chromatogram looks complicated, but structurally it's the simplest
thing in the lab: **two aligned columns of numbers** — an x-axis you set (wavelength,
time, m/z) and a y-axis you measure (absorbance, intensity).

Once you can hold those two columns in Python and plot one against the other, you can
look at any spectrum you'll ever record.

> **One idea to hold onto:** a spectrum is just paired x/y data — `x` you control,
> `y` you measure — kept the same length and in the same order.

**By the end of this notebook you will be able to:**

1. Store an x-axis and a y-axis as paired sequences and keep them aligned.
2. Use a NumPy **array** to do math on a whole spectrum at once.
3. Build a small, believable spectrum and plot it with labelled axes.

## 1. Two columns, side by side

A `list` holds a sequence of numbers in order. Here are a few wavelengths and the
absorbance measured at each. The two lists are *aligned*: the i-th wavelength goes
with the i-th absorbance.

In [ ]:
wavelength_nm = [500, 520, 540, 560, 580]   # x-axis: we chose these
absorbance = [0.10, 0.32, 0.71, 0.40, 0.12] # y-axis: we measured these

# zip walks the two lists together, pairing matching positions.
for wl, ab in zip(wavelength_nm, absorbance):
    print(wl, "nm  ->", ab, "AU")

# Alignment check: the two axes MUST be the same length.
print("aligned?", len(wavelength_nm) == len(absorbance))

That alignment check (`len(x) == len(y)`) is worth a habit — a mismatched axis is one
of the most common ways a plot comes out nonsense.

## 2. Why arrays: math on the whole spectrum at once

A plain list can't do arithmetic across all its values at once — `absorbance - 0.05`
would error. A **NumPy array** can. This matters because real preprocessing (like
subtracting a baseline) is exactly "do this to every point."

In [ ]:
import numpy as np

absorbance_arr = np.array(absorbance)   # turn the list into an array

# Subtract a constant baseline offset from EVERY point, in one line.
baseline = 0.05
corrected = absorbance_arr - baseline

print("raw      :", absorbance_arr)
print("corrected:", np.round(corrected, 3))

One line, every point. That whole-array style is the heart of scientific Python, and
Track 2 builds on it heavily.

## 3. Build a believable little spectrum

Let's make a fuller spectrum: a smooth wavelength axis, a single absorbance band
(a Gaussian peak) sitting on a small baseline, plus a touch of noise so it looks
measured. We fix a random *seed* so it's identical every run (your reproducibility
habit from 1.2).

In [ ]:
# A dense, evenly spaced wavelength axis from 450 to 650 nm (200 points).
wl = np.linspace(450, 650, 200)

# One Gaussian absorbance band centred at 540 nm.
peak_center_nm = 540.0
peak_height = 0.8
peak_width = 12.0   # controls how broad the band is
band = peak_height * np.exp(-((wl - peak_center_nm) ** 2) / (2 * peak_width ** 2))

# A small flat baseline, and a little reproducible noise.
rng = np.random.default_rng(seed=1)            # fixed seed -> identical every run
spectrum = band + 0.05 + rng.normal(0, 0.01, size=wl.size)

print("points:", wl.size, "| axis:", wl.min(), "-", wl.max(), "nm")

## 4. Plot it

Now the payoff: plot absorbance against wavelength, with labelled axes and units. A
spectrum you can *see* is a spectrum you can reason about.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(wl, spectrum)
ax.set_xlabel("wavelength (nm)")
ax.set_ylabel("absorbance (AU)")
ax.set_title("A simple simulated absorbance band")
plt.tight_layout()
plt.show()

## 5. Read a number back off the spectrum

`argmax` gives the **index** of the largest y-value; use it to look up the wavelength
of peak absorbance — the kind of number you'd actually report (lambda-max).

In [ ]:
peak_index = np.argmax(spectrum)          # position of the maximum
lambda_max = wl[peak_index]               # the wavelength there

print("peak absorbance at:", round(lambda_max, 1), "nm")
print("(we built the band at 540 nm, so this should land right next to it)")

## Key Takeaways

- A spectrum is **paired x/y sequences** of equal length, kept in the same order.
- A **NumPy array** lets you do math on the whole spectrum in one line — the basis of
  all preprocessing.
- `np.linspace` builds an even axis; `argmax` reads back the position of a peak.

## Practical Checklist

- [ ] Confirm `len(x) == len(y)` before plotting or doing math.
- [ ] Convert lists to arrays when you need whole-spectrum arithmetic.
- [ ] Always label both axes with quantity **and** unit.
- [ ] Fix a random seed whenever a step uses randomness.

## Common Mistakes

- Misaligned axes (different lengths, or sorted differently) producing a garbled plot.
- Trying to do arithmetic on a plain list instead of an array.
- Plotting without axis labels, so the figure can't be interpreted later.

## Next Lesson

**1.5 — Loops and Conditionals for Batch Thinking.** One spectrum is rarely the job.
Next we apply the same handling to many samples and flag the ones that need a look.